# 05 - Feature Engineering Pipeline

Notebook tổng hợp pipeline Feature Engineering cho dự án Data For Impact.

**3 nhóm features:**
1. **Baseline** — RFM, AOV, tenure, n_unique_days, avg_days_between_orders
2. **Temporal** — day_of_week_mode, recency_velocity, purchase_trend, lags, rolling means
3. **Synthetic** — purchase_intensity, loyalty_score, retention_score, customer_value_score, rfm_segment

In [1]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np

# Ensure project root is on path
PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_contract import run_data_contract_checks
from src.features.churn_feature_builder import build_churn_features
from src.features.forecast_feature_builder import build_forecast_features
from src.features.composite_feature_builder import build_composite_features

print("All modules imported successfully.")

All modules imported successfully.


## 1. Load Cleaned Data

In [2]:
DATA_PATH = PROJECT_ROOT / "data" / "cleaned" / "online_retail_cleaned_full.csv"
FEATURES_DIR = PROJECT_ROOT / "data" / "features"
FEATURES_DIR.mkdir(exist_ok=True)

df = pd.read_csv(DATA_PATH, parse_dates=["InvoiceDate"])

# Data contract check
contract_report = run_data_contract_checks(df)
print("Data Contract OK:")
for k, v in contract_report.items():
    print(f"  {k}: {v}")

print(f"\nShape: {df.shape}")
df.head()

Data Contract OK:
  rows: 797885
  columns: 28
  date_min: 2009-12-01 07:45:00
  date_max: 2011-12-09 12:50:00

Shape: (797885, 28)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,IsReturn,TotalValue,...,FirstPurchaseDate,CohortYear,CohortMonth,CohortQuarter,CohortMonthYear,CohortIndex,PositiveQuantity,PositiveTotalValue,InvoiceType,ProductCategory
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,False,83.4,...,2009-12-01 07:45:00,2009,12,4,2009-12,0,12,83.4,Sale,Decoration
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,False,81.0,...,2009-12-01 07:45:00,2009,12,4,2009-12,0,12,81.0,Sale,Decoration
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,False,81.0,...,2009-12-01 07:45:00,2009,12,4,2009-12,0,12,81.0,Sale,Decoration
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom,False,100.8,...,2009-12-01 07:45:00,2009,12,4,2009-12,0,48,100.8,Sale,Other
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,False,30.0,...,2009-12-01 07:45:00,2009,12,4,2009-12,0,24,30.0,Sale,Other


## 2. Build Baseline + Temporal Features (Churn Feature Builder)

Sử dụng snapshot_date = ngày cuối cùng trong dataset trừ inactivity_days để có ground truth churn label.

In [3]:
INACTIVITY_DAYS = 60

# Snapshot = max_date - inactivity_days so we have a future window for churn labeling
max_date = df["InvoiceDate"].max()
snapshot_date = max_date - pd.Timedelta(days=INACTIVITY_DAYS)
print(f"Max date in data: {max_date}")
print(f"Snapshot date:    {snapshot_date}")
print(f"Inactivity days:  {INACTIVITY_DAYS}")

churn_features = build_churn_features(
    df,
    snapshot_date=snapshot_date,
    inactivity_days=INACTIVITY_DAYS,
)

print(f"\nChurn features shape: {churn_features.shape}")
print(f"Columns: {churn_features.columns.tolist()}")
print(f"\nChurn rate: {churn_features['churn_label'].mean():.2%}")
churn_features.head(10)

Max date in data: 2011-12-09 12:50:00
Snapshot date:    2011-10-10 12:50:00
Inactivity days:  60

Churn features shape: (5581, 13)
Columns: ['Customer ID', 'snapshot_date', 'recency_days', 'frequency_90d', 'monetary_90d', 'avg_order_value_90d', 'return_rate_90d', 'tenure_days', 'n_unique_days', 'avg_days_between_orders', 'day_of_week_mode', 'recency_velocity', 'churn_label']

Churn rate: 62.78%


,Customer ID,snapshot_date,recency_days,frequency_90d,monetary_90d,avg_order_value_90d,return_rate_90d,tenure_days,n_unique_days,avg_days_between_orders,day_of_week_mode,recency_velocity,churn_label
0,12346,2011-10-10 12:50:00,265,0,0.00,0.0,0.0,665,11,8.543478,0,0.398496,1
1,12347,2011-10-10 12:50:00,69,22,584.91,26.586818,0.0,343,6,1.668712,1,0.201166,0
2,12348,2011-10-10 12:50:00,14,3,310.00,103.333333,0.0,377,5,7.240000,0,0.037135,1
3,12349,2011-10-10 12:50:00,347,0,0.00,0.0,0.0,675,4,3.075472,3,0.514074,0
4,12350,2011-10-10 12:50:00,249,0,0.00,0.0,0.0,249,1,0.000000,2,1.0,1
5,12351,2011-10-10 12:50:00,314,0,0.00,0.0,0.0,314,1,0.000000,0,1.0,1
6,12352,2011-10-10 12:50:00,11,32,632.50,19.765625,0.0,332,8,3.278351,1,0.033133,0
7,12353,2011-10-10 12:50:00,143,0,0.00,0.0,0.0,348,2,8.869565,2,0.41092,1
8,12354,2011-10-10 12:50:00,171,0,0.00,0.0,0.0,171,1,0.000000,3,1.0,1
9,12355,2011-10-10 12:50:00,153,0,0.00,0.0,0.0,507,2,10.382353,4,0.301775,1


### Nhóm 1 & 2: Baseline + Temporal Features Summary

In [4]:
baseline_cols = ["recency_days", "frequency_90d", "monetary_90d", "avg_order_value_90d",
                 "return_rate_90d", "tenure_days", "n_unique_days", "avg_days_between_orders"]
temporal_cols = ["day_of_week_mode", "recency_velocity"]

print("=" * 60)
print("NHÓM 1: BASELINE FEATURES")
print("=" * 60)
display(churn_features[baseline_cols].describe().round(2))

print("\n" + "=" * 60)
print("NHÓM 2: TEMPORAL FEATURES (from churn builder)")
print("=" * 60)
display(churn_features[temporal_cols].describe().round(2))

print("\nDay of week distribution (0=Mon, 6=Sun):")
print(churn_features["day_of_week_mode"].value_counts().sort_index())

NHÓM 1: BASELINE FEATURES


,recency_days,frequency_90d,monetary_90d,tenure_days,n_unique_days,avg_days_between_orders
count,5581.00,5581.00,5581.00,5581.00,5581.00,5581.00
mean,205.75,17.83,401.25,446.49,6.03,4.59
std,190.22,57.59,2431.00,199.21,10.39,12.93
min,0.00,0.00,-4287.63,0.00,1.00,0.00
25%,31.00,0.00,0.00,327.00,1.00,0.00
50%,144.00,0.00,0.00,495.00,3.00,1.70
75%,344.00,19.00,331.96,619.00,7.00,4.39
max,678.00,2133.00,92771.96,678.00,247.00,392.00



NHÓM 2: TEMPORAL FEATURES (from churn builder)


,day_of_week_mode
count,5581.00
mean,2.56
std,1.87
min,0.00
25%,1.00
50%,2.00
75%,4.00
max,6.00



Day of week distribution (0=Mon, 6=Sun):
day_of_week_mode
0     856
1     999
2     989
3    1199
4     752
5       1
6     785
Name: count, dtype: int64


## 3. Build Synthetic/Composite Features

In [5]:
composite_features = build_composite_features(churn_features)

synthetic_cols = ["purchase_intensity", "loyalty_score", "retention_score",
                  "customer_value_score", "rfm_segment"]

print("=" * 60)
print("NHÓM 3: SYNTHETIC/COMPOSITE FEATURES")
print("=" * 60)
display(composite_features[synthetic_cols[:4]].describe().round(4))

print("\nRFM Segment Distribution:")
print(composite_features["rfm_segment"].value_counts())

NHÓM 3: SYNTHETIC/COMPOSITE FEATURES


,purchase_intensity,loyalty_score,customer_value_score
count,5581.0000,5581.0000,5581.0000
mean,0.3289,0.4099,0.1627
std,3.3248,0.1070,0.0183
min,0.0000,0.3000,0.0001
25%,0.0000,0.3015,0.1578
50%,0.0000,0.3836,0.1578
75%,0.0516,0.5035,0.1634
max,156.0000,0.9978,0.6766



RFM Segment Distribution:
rfm_segment
Loyal                 1361
Lost                  1188
Champions              973
Hibernating            774
Need Attention         579
Potential Loyalist     457
At Risk                249
Name: count, dtype: int64


## 4. Build Forecast Features (Time-Series)

In [6]:
# Only use sales data (not returns) for forecasting
sales_df = df[df["IsReturn"] == False].copy()

forecast_features = build_forecast_features(sales_df)

forecast_cols = ["revenue", "lag_1", "lag_3", "rolling_mean_3",
                 "month_index", "month_number", "purchase_trend"]

print("=" * 60)
print("NHÓM 2: TEMPORAL FEATURES (from forecast builder)")
print("=" * 60)
print(f"Shape: {forecast_features.shape}")
print(f"Unique customers: {forecast_features['Customer ID'].nunique()}")
display(forecast_features[forecast_cols].describe().round(2))

NHÓM 2: TEMPORAL FEATURES (from forecast builder)
Shape: (58537, 9)
Unique customers: 5881


c:\Users\Admin\OneDrive\Documents\python\data for impact\.venv\Lib\site-packages\numpy\lib\_function_base_impl.py:4596: RuntimeWarning: invalid value encountered in subtract
  diff_b_a = b - a
c:\Users\Admin\OneDrive\Documents\python\data for impact\.venv\Lib\site-packages\pandas\core\nanops.py:1027: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


,revenue,lag_1,lag_3,rolling_mean_3,month_index,month_number,purchase_trend
count,58537.00,52656.00,44673.00,52656.00,58537.00,58537.00,30060.00
mean,296.82,274.34,279.62,293.09,8.09,6.63,inf
std,1461.27,1269.86,1268.60,1098.65,6.24,3.34,NaN
min,0.00,0.00,0.00,0.00,0.00,1.00,-1.00
25%,0.00,0.00,0.00,0.00,3.00,4.00,-1.00
50%,0.00,0.00,0.00,112.13,7.00,7.00,-0.05
75%,310.82,289.50,300.00,286.42,13.00,9.00,NaN
max,168469.60,75412.64,75412.64,41005.74,24.00,12.00,inf


## 5. Export Master Feature Table

In [7]:
# Save customer-level features (churn + composite)
customer_features_path = FEATURES_DIR / "customer_features.csv"
composite_features.to_csv(customer_features_path, index=False)
print(f"Saved customer features: {customer_features_path}")
print(f"  Shape: {composite_features.shape}")
print(f"  Columns: {composite_features.columns.tolist()}")

# Save forecast features (monthly time-series per customer)
forecast_features_path = FEATURES_DIR / "forecast_features.csv"
forecast_features.to_csv(forecast_features_path, index=False)
print(f"\nSaved forecast features: {forecast_features_path}")
print(f"  Shape: {forecast_features.shape}")

Saved customer features: c:\Users\Admin\OneDrive\Documents\python\data for impact\404_Not_Found\data\features\customer_features.csv
  Shape: (5581, 18)
  Columns: ['Customer ID', 'snapshot_date', 'recency_days', 'frequency_90d', 'monetary_90d', 'avg_order_value_90d', 'return_rate_90d', 'tenure_days', 'n_unique_days', 'avg_days_between_orders', 'day_of_week_mode', 'recency_velocity', 'churn_label', 'purchase_intensity', 'loyalty_score', 'retention_score', 'customer_value_score', 'rfm_segment']

Saved forecast features: c:\Users\Admin\OneDrive\Documents\python\data for impact\404_Not_Found\data\features\forecast_features.csv
  Shape: (58537, 9)


## 6. Feature Summary Report

In [8]:
print("=" * 70)
print("        FEATURE ENGINEERING SUMMARY")
print("=" * 70)

feature_summary = {
    "Nhóm 1: Baseline": baseline_cols,
    "Nhóm 2: Temporal (churn)": temporal_cols,
    "Nhóm 2: Temporal (forecast)": ["lag_1", "lag_3", "rolling_mean_3",
                                     "month_index", "month_number", "purchase_trend"],
    "Nhóm 3: Synthetic": synthetic_cols,
}

total = 0
for group, cols in feature_summary.items():
    print(f"\n{group}: ({len(cols)} features)")
    for c in cols:
        print(f"  • {c}")
    total += len(cols)

print(f"\n{'=' * 70}")
print(f"TOTAL: {total} features")
print(f"Customer-level features: {composite_features.shape[1]} columns, {composite_features.shape[0]} customers")
print(f"Forecast features: {forecast_features.shape[1]} columns, {forecast_features.shape[0]} rows")
print(f"{'=' * 70}")

        FEATURE ENGINEERING SUMMARY

Nhóm 1: Baseline: (8 features)
  • recency_days
  • frequency_90d
  • monetary_90d
  • avg_order_value_90d
  • return_rate_90d
  • tenure_days
  • n_unique_days
  • avg_days_between_orders

Nhóm 2: Temporal (churn): (2 features)
  • day_of_week_mode
  • recency_velocity

Nhóm 2: Temporal (forecast): (6 features)
  • lag_1
  • lag_3
  • rolling_mean_3
  • month_index
  • month_number
  • purchase_trend

Nhóm 3: Synthetic: (5 features)
  • purchase_intensity
  • loyalty_score
  • retention_score
  • customer_value_score
  • rfm_segment

TOTAL: 21 features
Customer-level features: 18 columns, 5581 customers
Forecast features: 9 columns, 58537 rows
